# Libraries

In [ ]:
import os
import time
import matplotlib.pyplot as plt
import torch
import tensorflow as tf

from models import generator_model, discriminator_model, NOISE_DIM

In [ ]:
dataset_path = "dataset.pt"
out_images_path = "img_output"
checkpoint_dir = "checkpoint_last"
os.makedirs(out_images_path, exist_ok=True)

# Converting torch tensor to tf tensor

In [ ]:
pytorch_tensor = torch.load(dataset_path)
tf_tensor = tf.convert_to_tensor(pytorch_tensor.numpy(), dtype=tf.float32)

# Reshaping the tensor

In [ ]:
tf_tensor = tf.reshape(tf_tensor, [-1, 256, 256, 1])

# Creating Dataset

In [ ]:
def create_dataloader(data_tensor, batch_size):
    dataset = tf.data.Dataset.from_tensor_slices(data_tensor)
    dataset = dataset.shuffle(buffer_size=len(data_tensor))
    dataset = dataset.batch(batch_size, drop_remainder=True)
    dataset = dataset.prefetch(tf.data.AUTOTUNE)
    return dataset

In [ ]:
batch_size = 64
dataloader = create_dataloader(tf_tensor, batch_size)

In [ ]:
dataloader

# Generator and Discriminator

Both models are defined in `models.py` so that this notebook and `5.generation_using_model.ipynb` use exactly the same architecture.

- **Generator:** a 512-d noise vector is reshaped to 1×1×512, expanded to 4×4 with a transposed convolution, then upsampled six times (conv → batchnorm → ReLU → upsample) to 256×256×1 with a `tanh` output.
- **Discriminator:** Gaussian noise on the input, then six conv → LeakyReLU → dropout → average-pool blocks down to 4×4×256, followed by two dense layers and a `sigmoid` output.

In [ ]:
generator = generator_model()
discriminator = discriminator_model()

In [ ]:
generator.summary()

In [ ]:
discriminator.summary()

# Noise Generation

In [ ]:
nz = tf.random.normal([1, NOISE_DIM])
generated_image = generator(nz, training=False)

In [ ]:
plt.imshow((generated_image[0, :, :, 0] + 1) / 2, cmap="gray")

# Defining Loss and Optimisers

## Discriminator

In [ ]:
# The discriminator already ends with a sigmoid, so it outputs probabilities, not logits
cross_entropy = tf.keras.losses.BinaryCrossentropy(from_logits=False)

Label smoothing with noisy labels: real images get a label in [0.95, 1.0] and fake images a label in [0.0, 0.05]. `tf.random` is used so new labels are drawn at every step inside the compiled `train_step`.

In [ ]:
def discriminator_loss(real_output, fake_output):
    real_values = tf.random.uniform(tf.shape(real_output), 0.95, 1.0)
    fake_values = tf.random.uniform(tf.shape(fake_output), 0.0, 0.05)

    real_loss = cross_entropy(real_values, real_output)
    fake_loss = cross_entropy(fake_values, fake_output)
    total_loss = real_loss + fake_loss
    return total_loss

In [ ]:
discriminator_optimizer = tf.keras.optimizers.Adam(1e-5)

## Generator

In [ ]:
def generator_loss(fake_output):
    return cross_entropy(tf.ones_like(fake_output), fake_output)

In [ ]:
generator_optimizer = tf.keras.optimizers.Adam(1e-5)

# Checkpoints Saving

In [ ]:
checkpoint_prefix = os.path.join(checkpoint_dir, "ckpt")
checkpoint = tf.train.Checkpoint(generator_optimizer=generator_optimizer,
                                 discriminator_optimizer=discriminator_optimizer,
                                 generator=generator,
                                 discriminator=discriminator)

# Training Loop

In [ ]:
num_examples_to_generate = 9

seed = tf.random.normal([num_examples_to_generate, NOISE_DIM])

In [ ]:
@tf.function
def train_step(images):
    noise = tf.random.normal([batch_size, NOISE_DIM])

    with tf.GradientTape() as gen_tape, tf.GradientTape() as disc_tape:
        generated_images = generator(noise, training=True)
        real_output = discriminator(images, training=True)
        fake_output = discriminator(generated_images, training=True)
        gen_loss = generator_loss(fake_output)
        disc_loss = discriminator_loss(real_output, fake_output)

    gradients_of_discriminator = disc_tape.gradient(disc_loss, discriminator.trainable_variables)
    discriminator_optimizer.apply_gradients(zip(gradients_of_discriminator, discriminator.trainable_variables))
    gradients_of_generator = gen_tape.gradient(gen_loss, generator.trainable_variables)
    generator_optimizer.apply_gradients(zip(gradients_of_generator, generator.trainable_variables))
    return gen_loss, disc_loss

The training loop runs one training step per batch, and every 50 epochs it saves a checkpoint and a grid of generated samples.

In [ ]:
def train(dataset, start_epoch, epochs):
    for epoch in range(start_epoch, start_epoch + epochs):
        start = time.time()
        for image_batch in dataset:
            gen_loss, disc_loss = train_step(image_batch)
        # Save the model every 50 epochs
        if (epoch + 1) % 50 == 0:
            checkpoint.save(file_prefix=checkpoint_prefix)
            generate_and_save_images(generator, epoch + 1, seed)
        print('Epoch {}: gen_loss {:.4f}, disc_loss {:.4f}, {:.1f} sec'.format(
            epoch + 1, float(gen_loss), float(disc_loss), time.time() - start))

    # Generate after the final epoch
    generate_and_save_images(generator, start_epoch + epochs, seed)

# Generate and save images samples

In [ ]:
def generate_and_save_images(model, epoch, test_input):
    predictions = model(test_input, training=False)
    fig = plt.figure(figsize=(10, 10))
    for i in range(predictions.shape[0]):
        plt.subplot(3, 3, i + 1)
        plt.imshow((predictions[i, :, :, 0] + 1) / 2, cmap="gray")
        plt.axis('off')
    plt.savefig(os.path.join(out_images_path, "image_at_epoch_{:04d}.png".format(epoch)))
    plt.show()

# Train the Model

## Get back to latest ckpt

Uncomment to resume from the last saved checkpoint, and set `last_trained_epoch` to the epoch it was saved at.

In [ ]:
# checkpoint.restore(tf.train.latest_checkpoint(checkpoint_dir))

## Model Training

In [ ]:
last_trained_epoch = 0
epochs_to_train = 200

train(dataloader, last_trained_epoch, epochs_to_train)